# AI Attitudes Survey Widgets

This notebook contains a number of 'widgets' (in other words, code examples) that you can use to analyze and describe the AI Attitudes Survey.

## Preliminaries

Run the code cell to load and clean the data and import necessary libraries. This code renames columns and drops unneeded columns.

Run this code **before** running any of the widgets in this notebook.

In [ ]:
from datetime import date
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from textblob import TextBlob, Word
from wordcloud import WordCloud

file = "/home/shared/AI_Attitudes.csv"
survey = pd.read_csv(file)

# Rename and drop columns
survey.columns=['ID', 'Start', 'End', 'Email', 'Name', 'Last Edit', 'Age', 'Gender', 'Excited', 'Why Excited', 'Concerned', 'Why Concerned', 'Use Freq', 'How Used', 'Tools', 'Report']
survey.drop(columns=['Email', 'Name', 'Last Edit', 'Report'], inplace=True)
survey

# Some recoding using np.select()
# Firstly, create a Net Excited-Concerned score (Net E-C) based on Excited - Concerned
# Then recode age into ascending numerical values so we can order output later
survey['Net E-C'] = survey['Excited'] - survey['Concerned']
conditions = [survey['Net E-C']>0, survey['Net E-C']==0, survey['Net E-C']<0]
values = ['More excited than concerned', 'Equally excited and concerned', 'More concerned than excited']
survey['Net E-C Cat'] = np.select(conditions, values, default='')
conditions = [survey['Age']=='Under 18', survey['Age']=='18-24', survey['Age']=='25-34', survey['Age']=='35-44', survey['Age']=='45-54', survey['Age']=='55 or over', survey['Age']=='Prefer not to say']
values = [0, 1, 2, 3, 4, 5, 6]
survey['Age Code'] = np.select(conditions, values, default=0)
survey

## 1. Summary Statistics

The tools below give you examples of how to create summary statistics using the `describe()` method for numerical data and how to compare summary statistics for different subgroups of the dataset. For example:

- `survey.describe()` will give you summary stats for all numerical values
- `survey[survey['Gender']=='Male'].describe()` will give you summary stats just for the people who gave their gender as male.
- `survey.groupby('Age')['Excited'].describe()` will give you summary stats by age group, just for the 'Excited' variable.

In [ ]:
# Overall summary stats
survey.describe()

In [ ]:
# Summary stats for a subgroup 
survey[survey['Gender']=='Male'].describe()

In [ ]:
# groupby() allows you to examine statistics split across categories
survey.groupby('Age')['Excited'].describe()

## 2. Crosstabs

A crosstab (short for cross tabulation) gives you a count of how many respondents fall into particular categories, based on two or more data columns.

For example, the crosstab below, gives you a table showing how many people fall into each combination of 'Gender' and 'Age' categories.

`pd.crosstab(survey['Gender'], survey['Age'])`

In [ ]:
# A crosstab allows you to compare one column based on categories in other columns
# In this example, we can see that there are 12 males between the ages of 18-24, for example
pd.crosstab(survey['Gender'], survey['Age'],margins=True, margins_name="Total")

## 3. Histograms

Histograms allow you to see the distribution of numerical data. The horizontal axis shows values for the numerical data and the vertical axes gives you frequencies.

In [ ]:
# This first histogram shows the distribution of excited scores
plt.hist(survey['Excited'], bins=np.arange(1, 11))
plt.title("Histogram of 'Excited' Scores - All Respondents")
plt.show()

# This does the same thing, but just for males
plt.hist(survey[survey['Gender']=='Male']['Excited'], bins=np.arange(1, 11))
plt.title("Histogram of 'Excited' Scores - Male Respondents")
plt.show()

## 4. Using Value Counts

Suppose we want to know how many people are in each age category. The `value_counts()` method allows use to do this. 

A convenient way to use `value_counts()` is to put the result into a new dataframe. This allows you to graph the data easily.

Sometimes the counts are not sorted in the way you want, so you need to find a way to sort them.

In [ ]:
#  This code puts the 'Age' value counts into a new dataframe
# Because value_counts returns values in no particular order, we sort by 'Age Code' to order the data correctly
age_distr = survey['Age Code'].value_counts()
age_distr = pd.DataFrame(age_distr)
age_distr.reset_index(inplace=True)
age_distr.sort_values(by='Age Code', inplace=True)
age_distr

In [ ]:
# Now we can create a bar chart using the data above
# We just need to create a list of categories for age that correspond with our value counts
age_cats = ['Under 18', '18-24', '25-34', '35-44', '45-54', 'Over 55', 'NA']
plt.barh(age_cats, age_distr['count'])
plt.title("Age Distribution - All Respondents")
plt.show()

## 5. Comparing Two or More Groups

Some times we want to look at one variable across two or more groups. For example, we might want to look at the distribution of ages for males and females.

Depending on the type of data, there are several ways to do this:

- A bar chart with two or more sets of bars
- Separate, side-by-side plots
- A side-by-side boxplot (for numerical data)

The examples below show how to do all of these.

### 5a. Bar Charts

Bar charts are good for displaying categorical data. You can display categorical data for two groups on the same chart using multiple bars.

In [ ]:
# Bar Charts with two sets of columns

# Split whole dataset into 'Male' and "Female' sections
survey_m = survey[survey['Gender']=='Male']
survey_f = survey[survey['Gender']=='Female']

# Create a value count for "Age Code" for male respondents
ages_m = survey_m['Age Code'].value_counts()
ages_m = pd.DataFrame(ages_m)
ages_m.reset_index(inplace=True)
ages_m.sort_values(by='Age Code', inplace=True)
ages_m.loc[len(ages_m)]=[6, 0]
ages_m

In [ ]:
# Create a value count for 'Age Code' for female respondents
ages_f= survey_f['Age Code'].value_counts()
ages_f = pd.DataFrame(ages_f)
ages_f.reset_index(inplace=True)
ages_f.sort_values(by='Age Code', inplace=True)
ages_f

In [ ]:
# Now we can create a single bar chart that shows the distribution of ages for males and females
width = 0.35  # the width of the bars

x = np.arange(len(age_cats))
fig, ax = plt.subplots()
rects1 = ax.bar(x - width/2, ages_m['count'], width, label='Male')
rects2 = ax.bar(x + width/2, ages_f['count'], width, label='Female')

ax.set_ylabel('Frequency')
ax.set_xlabel('Age')
ax.set_title('Demographic Profile of Respondents')
ax.set_xticks(x, age_cats)
ax.legend()
plt.show()

### 5b. Side By Side Plots
|
An alternative, that may be better in some cases, is to display two separate graphs or charts next to each other. The Python code to do this is shown below.

In [ ]:
# Create 1 row and 2 columns
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

# Plot on the first (left) axis
ax1.bar( age_cats,ages_m['count'])
ax1.set_title("Age Distribution - Males")

# Plot on the second (right) axis
ax2.bar( age_cats,ages_f['count'])
ax2.set_title("Age Distribution - Females")

plt.tight_layout() # Prevents overlapping labels
plt.show()


### 5c. Box Plots

Side-by-side box plots are a good way to compare numerical data for different groups. With box plots you can easily compare more than two groups.

In [ ]:
# Here we compare the net excited-concerned score for males and females
plt.boxplot([survey_m['Net E-C'], survey_f['Net E-C']])
plt.show()

## 6. Using Group By

If you just want numerical statistics based on groups, you can also use the `.groupby()` method. For example, to work out the mean 'excited' score for different age groups, you would write the following:
```
survey.groupby('Age')['Excited'].mean()
```


In [ ]:
# Get mean 'Excited' score by age group
survey.groupby('Age')['Excited'].mean()

## 7. Saving Plots as an image file

If you want to save a plot, you can use the follow command.
```
plt.savefig('my_plot.png')
```

This assumes you have imported `matplotlib.pyplot` as `plt`. It will save the file as `my_plot.png.` You can choose any filename you want.